# Double-Gauss optimisation With Differential Evolution

This notebook demonstrates a mixed-variable optimisation workflow for a Double-Gauss lens using `DoubleGaussObjective` from `examples/double_gauss_objective.py`.

We optimize a packed parameter vector

$$
\boldsymbol{\theta} = [\mathbf{x}, \mathbf{m}],
$$

where:
- $\mathbf{x}\in\mathbb{R}^{n_x}$ are continuous design variables (curvatures and spacings),
- $\mathbf{m}\in\mathbb{Z}^{n_m}$ are integer material IDs.

The objective is the optical loss

$$
\min_{\boldsymbol{\theta}}\; f(\boldsymbol{\theta})
\quad\text{subject to}\quad
\mathbf{l} \le \boldsymbol{\theta} \le \mathbf{u}.
$$

The cells below cover setup, baseline evaluation, runtime benchmarking, and a SciPy Differential Evolution run.



In [1]:
from __future__ import annotations

from importlib import reload

import jax
import numpy as np
import double_gauss_objective


import lensgopt.optics.shapes as shapes
import lensgopt.parsers.lens_creation as lens_creation
import lensgopt.parsers.zmax_fmt as zmax_fmt

reload(shapes)
reload(zmax_fmt)
reload(lens_creation)
reload(double_gauss_objective)

jax.config.update("jax_enable_x64", True)

## 1) Build the objective and variable bounds

This cell instantiates `DoubleGaussObjective` and extracts optimisation bounds.

Key outputs:
- `n_x`: number of continuous variables,
- `n_m`: number of integer material variables,
- `n_theta = n_x + n_m`: total dimensionality,
- `lb`, `ub`: lower/upper bound vectors for all packed variables.

These bounds define the feasible search box used by Differential Evolution.



In [2]:
objective = double_gauss_objective.DoubleGaussObjective()
lb, ub = objective.bounds()

print(f"n_continuous={objective.n_x}, n_integer={objective.n_m}, n_theta={objective.n_theta}")
print("integer indices:", objective.integer_indices)
print("bounds shape:", lb.shape, ub.shape)


n_continuous=18, n_integer=6, n_theta=24
integer indices: [18, 19, 20, 21, 22, 23]
bounds shape: (24,) (24,)


## 2) Evaluate the baseline design and local derivatives

We initialize parameters from the template lens and verify two equivalent objective interfaces:
- `objective_theta(theta)` for packed variables,
- `objective_cont_int(x_cont, x_ids)` for split variables.

We also compute first- and second-order quantities with respect to continuous variables:

$$
\nabla f(\mathbf{x}), \qquad H(\mathbf{x}) = \nabla^2 f(\mathbf{x}).
$$

Finally, we visualize the baseline optical layout and ray paths.



In [3]:
x0_cont, x0_ids = objective.init_from_templates()
theta0 = objective.pack_theta(x0_cont, x0_ids)

loss0_theta = objective.objective_theta(theta0)
loss0_split = objective.objective_cont_int(x0_cont, x0_ids)
assert np.abs(loss0_theta - loss0_split) < 1e-10
grad0_cont = objective.gradient_cont_int(x0_cont, x0_ids)
H0 = objective.hessian_cont_int(x0_cont, x0_ids)

print(f"double gauss lens loss: {loss0_theta:.8f}")
print("grad continuous shape:", grad0_cont.shape)
print("hessian shape:", H0.shape)
print("grad continuous norm:", np.linalg.norm(grad0_cont))
print("hessian eigvals:", np.linalg.eigvalsh(H0))

fig0, ax0, _ = objective.visualize(
    theta=theta0,
    use_latex=True,
    is_post_optimized=False,
)

double gauss lens loss: 0.00055854
grad continuous shape: (18,)
hessian shape: (18, 18)
grad continuous norm: 0.0025539550838631687
hessian eigvals: [-5.89534345e-05 -5.17591433e-07 -1.23298425e-20  4.15389763e-20
  2.38513330e-09  2.97908738e-08  3.14805851e-07  4.58926944e-07
  8.35504288e-07  4.69822952e-06  9.86834765e-06  2.68564935e-05
  7.40509762e-05  1.57327320e-04  4.52613940e-04  3.88947733e-03
  5.26259894e-03  7.07893355e+01]


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 944.88x531.495 with 1 Axes>

## 3) Benchmark objective and gradient runtime

This section reports average runtime after JAX warm-up (XLA compilation).

Measured statistic:

$$
\bar t = \frac{1}{N}\sum_{k=1}^{N}(t_k^{\mathrm{end}}-t_k^{\mathrm{start}}).
$$

This gives a practical estimate of optimisation cost per function/gradient call.



In [4]:
import time
from typing import Any, Callable

reload(double_gauss_objective)


def measure_avg_runtime(
    fn: Callable[..., Any], runs: int = 1000, *args, **kwargs
) -> float:
    total = 0.0
    for _ in range(runs):
        start = time.perf_counter()
        fn(*args, **kwargs)
        total += time.perf_counter() - start
    return total / runs


objective = double_gauss_objective.DoubleGaussObjective()

x0_cont, x0_ids = objective.init_from_templates()
theta0 = objective.pack_theta(x0_cont, x0_ids)

# Compile XLA
objective.objective_theta(theta0)
objective.gradient_cont_int(x0_cont, x0_ids)

loss_t = measure_avg_runtime(
    objective._f_loss, 1000, x0_cont, x0_ids
)
grad_t = measure_avg_runtime(objective._f_grad, 1000, x0_cont, x0_ids)

print(
    f"Average time for loss computation: {1000*loss_t:.1f} [ms]",
)
print(
    f"Average time for grad computation: {1000*grad_t:.1f} [ms]",
)

Average time for loss computation: 0.9 [ms]
Average time for grad computation: 24.2 [ms]


## 4) Optimize with SciPy Differential Evolution

This cell runs global optimisation with `scipy.optimize.differential_evolution`.

Implementation details:
- `bounds` are taken from the objective setup,
- `integrality` marks material-ID coordinates as integer dimensions,
- `workers=1` keeps evaluation single-process (typically safer with JAX in notebooks),
- `polish=False` disables local post-processing so results come from DE alone.

After optimisation, we project to valid bounds/integer IDs, report the best loss, and visualize the optimized design.



In [5]:
# Differential Evolution example
from scipy.optimize import differential_evolution


de_bounds = list(zip(lb, ub))
de_integrality = np.zeros(objective.n_theta, dtype=bool)
de_integrality[objective.integer_indices] = True

de_result = differential_evolution(
    objective.objective_theta,
    bounds=de_bounds,
    integrality=de_integrality,
    maxiter=100,
    popsize=10,
    seed=42,
    polish=False,
    workers=1,
)

theta_de_best = objective.project_theta(np.asarray(de_result.x, dtype=float), lb=lb, ub=ub)
loss_de_best = objective.objective_theta(theta_de_best)

print(f"DE status: {de_result.message}")
print(f"DE evaluations: {de_result.nfev}")
print(f"DE best loss: {loss_de_best:.8f}")
fig_best, ax_best, plotted_loss = objective.visualize(
    theta=theta_de_best,
    loss_value=loss_de_best,
    use_latex=True,
)


DE status: Maximum number of iterations has been exceeded.
DE evaluations: 24240
DE best loss: 0.11184144


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 944.88x531.495 with 1 Axes>